<a href="https://colab.research.google.com/github/atomicSteiner/HealthcareSBERT/blob/main/EmbeddingsBuilder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path_drive = "/content/drive/MyDrive/NLP/"

In [ ]:
from datasets import load_dataset
import pandas as pd
import os
import numpy as np

In [ ]:
# Load OHSUMED
ds = load_dataset("community-datasets/ohsumed")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ohsumed/train-00000-of-00001.parquet:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

ohsumed/test-00000-of-00001.parquet:   0%|          | 0.00/181M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/54709 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/293855 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['seq_id', 'medline_ui', 'mesh_terms', 'title', 'publication_type', 'abstract', 'author', 'source'],
        num_rows: 54709
    })
    test: Dataset({
        features: ['seq_id', 'medline_ui', 'mesh_terms', 'title', 'publication_type', 'abstract', 'author', 'source'],
        num_rows: 293855
    })
})


In [ ]:
#switching train and test split
df_test = pd.read_csv(path_drive + "ohsumed_cleaned_train.csv")
df_train = pd.read_csv(path_drive + "ohsumed_cleaned_test.csv")

In [ ]:
abstracts = df_test['abstract'].tolist()

preprocessing


In [ ]:
# MeSH parsing, prepare the data
def parse_mesh_improved(mesh_str):
    if not isinstance(mesh_str, str):
        return []

    terms = mesh_str.split(';')  # split by semicolon
    clean_terms = []

    for t in terms:
        t = t.strip()            # remove whitespace
        if t == '':
            continue
        # keep only main term (before '/')
        t = t.split('/')[0].strip()
        clean_terms.append(t)

    return clean_terms

df_train['mesh_terms'] = df_train['mesh_terms'].apply(parse_mesh_improved)
df_test['mesh_terms'] = df_test['mesh_terms'].apply(parse_mesh_improved)

In [ ]:
import random
from tqdm import tqdm

mesh_to_abstracts = {}

for _, row in df_train.iterrows():
    abstract = row["abstract"]
    for mesh in row["mesh_terms"]:
        mesh_to_abstracts.setdefault(mesh, []).append(abstract)

positive_pairs = []
max_pairs_per_mesh = 20   # memory-safe

for abstracts in mesh_to_abstracts.values():
    if len(abstracts) < 2:
        continue
    sample = random.sample(abstracts, min(len(abstracts), max_pairs_per_mesh))
    for i in range(len(sample)):
        for j in range(i+1, len(sample)):
            positive_pairs.append((sample[i], sample[j]))

print("Positive pairs:", len(positive_pairs))


Positive pairs: 1779318


In [ ]:
from sentence_transformers import InputExample

# 1) positive val pairs (sampled)
pos_val = random.sample(positive_pairs, min(2500, len(positive_pairs)))

# Build dict for fast mesh lookup
abstract_to_mesh = {
    row["abstract"]: set(row["mesh_terms"])
    for _, row in df_train.iterrows()
}

abstracts_list = list(abstract_to_mesh.keys())

# 2) negative val pairs
neg_val = set()
while len(neg_val) < len(pos_val):
    a1, a2 = random.sample(abstracts_list, 2)
    if len(abstract_to_mesh[a1].intersection(abstract_to_mesh[a2])) == 0:
        neg_val.add((a1, a2))

neg_val = list(neg_val)

print("Validation positives:", len(pos_val))
print("Validation negatives:", len(neg_val))

# Convert
val_examples = []
for a1, a2 in pos_val:
    val_examples.append(InputExample(texts=[a1, a2], label=1.0))
for a1, a2 in neg_val:
    val_examples.append(InputExample(texts=[a1, a2], label=0.0))

Validation positives: 2500
Validation negatives: 2500


In [ ]:
# Optional: sample a subset if still too large
limit = 100000
if len(positive_pairs) > limit:
    subgroup_positive_pairs = random.sample(positive_pairs, limit)
    print(f"Subsampled to {len(subgroup_positive_pairs)} training examples.")

Subsampled to 100000 training examples.


In [ ]:
# TRAIN EXAMPLES only positives
train_examples = [InputExample(texts=[a1, a2]) for a1, a2 in subgroup_positive_pairs]

In [ ]:
from sentence_transformers import SentenceTransformer, SentencesDataset, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("all-MiniLM-L6-v2")

# train
train_dataset = SentencesDataset(train_examples, model)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=4)

train_loss = losses.MultipleNegativesRankingLoss(model)

# validation
sentences1 = [ex.texts[0] for ex in val_examples]
sentences2 = [ex.texts[1] for ex in val_examples]
labels = [ex.label for ex in val_examples]


In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator(sentences1, sentences2, labels)

In [ ]:
import os


num_epochs = 4
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    evaluation_steps=1000,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path="fine_tuned_sbert_ohsumed",
    optimizer_params={"lr": 1.5e-5, "eps": 1e-6, "weight_decay": 0.01},
    use_amp=True,
    save_best_model=True
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
1000,0.566500,No log,0.721190,0.744514
2000,0.585700,No log,0.725675,0.747845
3000,0.546300,No log,0.731417,0.752114
4000,0.522900,No log,0.737570,0.757464
5000,0.559100,No log,0.730715,0.750761
6000,0.549400,No log,0.731722,0.749823
7000,0.562400,No log,0.742447,0.759114
8000,0.555500,No log,0.736528,0.751531
9000,0.545200,No log,0.729527,0.748590
10000,0.523700,No log,0.732433,0.750858


In [ ]:
from google.colab import files
!zip -r fine_tuned_sbert_ohsumed.zip fine_tuned_sbert_ohsumed
files.download('fine_tuned_sbert_ohsumed.zip')

  adding: fine_tuned_sbert_ohsumed/ (stored 0%)
  adding: fine_tuned_sbert_ohsumed/vocab.txt (deflated 53%)
  adding: fine_tuned_sbert_ohsumed/README.md (deflated 69%)
  adding: fine_tuned_sbert_ohsumed/model.safetensors (deflated 8%)
  adding: fine_tuned_sbert_ohsumed/1_Pooling/ (stored 0%)
  adding: fine_tuned_sbert_ohsumed/1_Pooling/config.json (deflated 59%)
  adding: fine_tuned_sbert_ohsumed/sentence_bert_config.json (deflated 9%)
  adding: fine_tuned_sbert_ohsumed/tokenizer.json (deflated 71%)
  adding: fine_tuned_sbert_ohsumed/2_Normalize/ (stored 0%)
  adding: fine_tuned_sbert_ohsumed/tokenizer_config.json (deflated 73%)
  adding: fine_tuned_sbert_ohsumed/config_sentence_transformers.json (deflated 40%)
  adding: fine_tuned_sbert_ohsumed/eval/ (stored 0%)
  adding: fine_tuned_sbert_ohsumed/eval/similarity_evaluation_results.csv (deflated 38%)
  adding: fine_tuned_sbert_ohsumed/config.json (deflated 47%)
  adding: fine_tuned_sbert_ohsumed/modules.json (deflated 62%)
  adding: fi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Baseline and FineTuned Model Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
last_run_path = "/content/drive/MyDrive/NLP/"

In [ ]:
fine_tuned_model = SentenceTransformer(last_run_path + 'fine_tuned_sbert_ohsumed/fine_tuned_sbert_ohsumed')

In [ ]:
fine_tuned_embeddings= fine_tuned_model.encode(abstracts, batch_size=16, device='cuda', show_progress_bar=True, convert_to_numpy=True)
np.save('fine_tuned_embeddings.npy', fine_tuned_embeddings)

Batches:   0%|          | 0/2306 [00:00<?, ?it/s]

In [ ]:
#download embeddings for offline use
!zip -r fine_tuned_embeddings.zip fine_tuned_embeddings.npy
from google.colab import files
files.download('fine_tuned_embeddings.zip')

  adding: fine_tuned_embeddings.npy (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
baseline_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
baseline_embeddings= baseline_model.encode(abstracts, batch_size=16, device='cuda', show_progress_bar=True, convert_to_numpy=True)
np.save('baseline_embeddings.npy', baseline_embeddings)

Batches:   0%|          | 0/2306 [00:00<?, ?it/s]

In [ ]:
#download embeddings for offline use
!zip -r baseline_embeddings.zip baseline_embeddings.npy
from google.colab import files
files.download('baseline_embeddings.zip')

  adding: baseline_embeddings.npy (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>